## TFT Baseline Walkthrough

In [ ]:
import sys; sys.path.insert(0, '..')
import warnings; warnings.filterwarnings('ignore')
import torch
import mlflow
from pathlib import Path

from src.data.loader import RossmannDataLoader
from src.data.preprocessor import RossmannPreprocessor
from src.data.feature_engineer import FeatureEngineer
from src.models.tft_dataset import build_tft_dataset
from src.models.tft_model import TFTForecaster
from src.models.tft_trainer import TFTTrainer
from src.models.tft_interpretability import TFTInterpreter

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
mlflow.set_tracking_uri('http://localhost:5000')
mlflow.set_experiment('retail_tft_forecasting')

In [ ]:
loader = RossmannDataLoader('../data/raw')
train_df, _ = loader.load()
pre = RossmannPreprocessor()
train_df = pre.fit_transform(train_df)
engineer = FeatureEngineer()
train_df = engineer.transform(train_df)

Path('../data/processed').mkdir(parents=True, exist_ok=True)
train_df.to_parquet('../data/processed/train_featured.parquet', index=False)
print(train_df.shape)

In [ ]:
train_dataset = build_tft_dataset(train_df, config_path='../configs/tft_config.yaml')
val_dataset = build_tft_dataset(train_df, config_path='../configs/tft_config.yaml', training=False, training_dataset=train_dataset)

forecaster = TFTForecaster('../configs/tft_config.yaml', train_dataset)
model = forecaster.build()

print(f'Train samples: {len(train_dataset)}')
print(f'Val samples: {len(val_dataset)}')

In [ ]:
from src.models.lr_finder import find_optimal_lr
train_loader = train_dataset.to_dataloader(train=True, batch_size=64, num_workers=0)
val_loader = val_dataset.to_dataloader(train=False, batch_size=128, num_workers=0)
lr_suggestion = find_optimal_lr(model, train_loader, val_loader, num_training_steps=50)
print(f'Suggested LR: {lr_suggestion:.2e}')

In [ ]:
trainer = TFTTrainer('../configs/tft_config.yaml')
with mlflow.start_run(run_name='tft_baseline_notebook'):
    trainer.fit(model, train_dataset, val_dataset)
print('Training complete')

In [ ]:
interpreter = TFTInterpreter(model, train_dataset)
vi_df = interpreter.variable_importance(val_loader)
print(vi_df.head(15))

In [ ]:
attn = interpreter.attention_weights(val_loader, n_samples=64)
profile = interpreter.seasonal_attention_profile(attn, window=7)
print(attn.shape, profile.shape)